this notebook runs proxy generation and verifies the output proxies

The three drop downs "freitsh timing templates", "fretish condition-timing templates", "PSP templates" consist of the fretish/PSP templates in LTL

Execute one of the cells to select the templates to generate proxies for and then execute all cells in "run proxy generation"

In [ ]:
import spot
from spot.jupyter import display_inline
import spot_utils
import nusmv_utils
import itertools
from tqdm import tqdm
import buddy
import time
import numpy as np
import collections

import proxy_gen

## fretish timing templates

In [ ]:
all_f_list = [['r',
  '(X r)',
  '((G[0,2] r) | (G r))',
  '(((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))',
  '(F[0,2] r)',
  '(s V (r | s))',
  '(r V (! s))',
  '(G r)',
  '(G (! r))',
  '(F r)'],
 ['(G (c1 -> r))',
  '(G (c1 -> (X r)))',
  '(G (c1 -> ((G[0,2] r) | (G r))))',
  '(G (c1 -> (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))',
  '(G (c1 -> (F[0,2] r)))',
  '(G (c1 -> (s V (r | s))))',
  '(G (c1 -> (r V (! s))))',
  '(G (c1 -> (G r)))',
  '(G (c1 -> (G (! r))))',
  '(G (c1 -> (F r)))'],
 ['((G (((! c2) & (X c2)) -> (X r))) & (c2 -> r))',
  '((G (((! c2) & (X c2)) -> (X (X r)))) & (c2 -> (X r)))',
  '((G (((! c2) & (X c2)) -> (X ((G[0,2] r) | (G r))))) & (c2 -> ((G[0,2] r) | (G r))))',
  '((G (((! c2) & (X c2)) -> (X (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))) & (c2 -> (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))',
  '((G (((! c2) & (X c2)) -> (X (F[0,2] r)))) & (c2 -> (F[0,2] r)))',
  '((G (((! c2) & (X c2)) -> (X (s V (r | s))))) & (c2 -> (s V (r | s))))',
  '((G (((! c2) & (X c2)) -> (X (r V (! s))))) & (c2 -> (r V (! s))))',
  '((G (((! c2) & (X c2)) -> (X (G r)))) & (c2 -> (G r)))',
  '((G (((! c2) & (X c2)) -> (X (G (! r))))) & (c2 -> (G (! r))))',
  '((G (((! c2) & (X c2)) -> (X (F r)))) & (c2 -> (F r)))'],
 ['((G ((! ((! m1) & (X m1))) | (X r))) & (m1 -> r))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))) & (m1 -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((G[0,2] r) | ((m1 & (X (! m1))) V r))))) & (m1 -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))',
  '((G ((! ((! m1) & (X m1))) | (X (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))) & (m1 -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))) & (m1 -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))) & (m1 -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))',
  '((G ((! ((! m1) & (X m1))) | (X (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))) & (m1 -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))',
  '(G (m1 -> r))',
  '(G (m1 -> (! r)))',
  '((G ((! ((! m1) & (X m1))) | (X ((! (m1 & (X (! m1)))) U r)))) & (m1 -> ((! (m1 & (X (! m1)))) U r)))'],
 ['((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> r))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> r))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> ((m1 & (X (! m1))) V r)))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> ((m1 & (X (! m1))) V r)))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> ((m1 & (X (! m1))) V (! r))))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> ((m1 & (X (! m1))) V (! r))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c1 -> ((! (m1 & (X (! m1)))) U r)))))) & (m1 -> ((m1 & (X (! m1))) V (c1 -> ((! (m1 & (X (! m1)))) U r)))))'],
 ['((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X r) & (! (m1 & (X (! m1))))))) & (c2 -> r))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X r) & (! (m1 & (X (! m1))))))) & (c2 -> r))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))) & (! (m1 & (X (! m1))))))) & (c2 -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))) & (! (m1 & (X (! m1))))))) & (c2 -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((G[0,2] r) | ((m1 & (X (! m1))) V r))) & (! (m1 & (X (! m1))))))) & (c2 -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((G[0,2] r) | ((m1 & (X (! m1))) V r))) & (! (m1 & (X (! m1))))))) & (c2 -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))) & (! (m1 & (X (! m1))))))) & (c2 -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))) & (! (m1 & (X (! m1))))))) & (c2 -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))))) & (c2 -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))))) & (c2 -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))) & (! (m1 & (X (! m1))))))) & (c2 -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))) & (! (m1 & (X (! m1))))))) & (c2 -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))) & (! (m1 & (X (! m1))))))) & (c2 -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))) & (! (m1 & (X (! m1))))))) & (c2 -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) V r)) & (! (m1 & (X (! m1))))))) & (c2 -> ((m1 & (X (! m1))) V r)))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) V r)) & (! (m1 & (X (! m1))))))) & (c2 -> ((m1 & (X (! m1))) V r)))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) V (! r))) & (! (m1 & (X (! m1))))))) & (c2 -> ((m1 & (X (! m1))) V (! r))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) V (! r))) & (! (m1 & (X (! m1))))))) & (c2 -> ((m1 & (X (! m1))) V (! r))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((! (m1 & (X (! m1)))) U r)) & (! (m1 & (X (! m1))))))) & (c2 -> ((! (m1 & (X (! m1)))) U r)))))) & (m1 -> (((m1 & (X (! m1))) V (((! c2) & ((X c2) & (! (m1 & (X (! m1)))))) -> ((X ((! (m1 & (X (! m1)))) U r)) & (! (m1 & (X (! m1))))))) & (c2 -> ((! (m1 & (X (! m1)))) U r)))))'],
 ['((G ((! (m2 & (X (! m2)))) | (X r))) & ((! m2) -> r))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))) & ((! m2) -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((G[0,2] r) | (((! m2) & (X m2)) V r))))) & ((! m2) -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))) & ((! m2) -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))) & ((! m2) -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))) & ((! m2) -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))',
  '((G ((! (m2 & (X (! m2)))) | (X (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))) & ((! m2) -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))',
  '(G ((! m2) -> r))',
  '(G (r -> m2))',
  '((G ((! (m2 & (X (! m2)))) | (X ((! ((! m2) & (X m2))) U r)))) & ((! m2) -> ((! ((! m2) & (X m2))) U r)))'],
 ['((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> r))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> r))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> (((! m2) & (X m2)) V r)))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> (((! m2) & (X m2)) V r)))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> (((! m2) & (X m2)) V (! r))))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> (((! m2) & (X m2)) V (! r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c1 -> ((! ((! m2) & (X m2))) U r)))))) & ((! m2) -> (((! m2) & (X m2)) V (c1 -> ((! ((! m2) & (X m2))) U r)))))'],
 ['((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X r) & (! ((! m2) & (X m2)))))) & (c2 -> r))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X r) & (! ((! m2) & (X m2)))))) & (c2 -> r))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))) & (! ((! m2) & (X m2)))))) & (c2 -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))) & (! ((! m2) & (X m2)))))) & (c2 -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X ((G[0,2] r) | (((! m2) & (X m2)) V r))) & (! ((! m2) & (X m2)))))) & (c2 -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X ((G[0,2] r) | (((! m2) & (X m2)) V r))) & (! ((! m2) & (X m2)))))) & (c2 -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))) & (! ((! m2) & (X m2)))))) & (c2 -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))) & (! ((! m2) & (X m2)))))) & (c2 -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))))) & (c2 -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))))) & (c2 -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))) & (! ((! m2) & (X m2)))))) & (c2 -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))) & (! ((! m2) & (X m2)))))) & (c2 -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))) & (! ((! m2) & (X m2)))))) & (c2 -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))) & (! ((! m2) & (X m2)))))) & (c2 -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) V r)) & (! ((! m2) & (X m2)))))) & (c2 -> (((! m2) & (X m2)) V r)))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) V r)) & (! ((! m2) & (X m2)))))) & (c2 -> (((! m2) & (X m2)) V r)))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) V (! r))) & (! ((! m2) & (X m2)))))) & (c2 -> (((! m2) & (X m2)) V (! r))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) V (! r))) & (! ((! m2) & (X m2)))))) & (c2 -> (((! m2) & (X m2)) V (! r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X ((! ((! m2) & (X m2))) U r)) & (! ((! m2) & (X m2)))))) & (c2 -> ((! ((! m2) & (X m2))) U r)))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c2) & ((X c2) & (! ((! m2) & (X m2))))) -> ((X ((! ((! m2) & (X m2))) U r)) & (! ((! m2) & (X m2)))))) & (c2 -> ((! ((! m2) & (X m2))) U r)))))'],
 ['(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X r))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (X r)))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G[0,2] r) | (G r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (F[0,2] r)))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (s V (r | s))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (r V (! s))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G r)))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (! r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (F r)))) | (G (! (m3 & (X (! m3))))))'],
 ['(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> (X r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> ((G[0,2] r) | (G r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> (F[0,2] r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> (s V (r | s))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> (r V (! s))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> (G r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> (G (! r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c1 -> (F r)))))) | (G (! (m3 & (X (! m3))))))'],
 ['(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X r))) & (c2 -> r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X (X r)))) & (c2 -> (X r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X ((G[0,2] r) | (G r))))) & (c2 -> ((G[0,2] r) | (G r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))) & (c2 -> (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X (F[0,2] r)))) & (c2 -> (F[0,2] r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X (s V (r | s))))) & (c2 -> (s V (r | s))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X (r V (! s))))) & (c2 -> (r V (! s))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X (G r)))) & (c2 -> (G r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X (G (! r))))) & (c2 -> (G (! r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c2) & (X c2)) -> (X (F r)))) & (c2 -> (F r)))))) | (G (! (m3 & (X (! m3))))))'],
 ['(r | m4)',
  '((((! m4) & (X m4)) | ((X r) & (! ((! m4) & (X m4))))) | m4)',
  '(((G[0,2] r) | (((! m4) & (X m4)) V r)) | m4)',
  '((((G[0,2] (! r)) | (((! m4) & (X m4)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m4) & (X m4))))) | m4)',
  '(((F[0,2] r) | (F[0,1] ((! m4) & (X m4)))) | m4)',
  '(((! (((! s) & (! ((! m4) & (X m4)))) U ((! r) & (! s)))) | (((r & ((! m4) & (X m4))) | ((! m4) & (X m4))) V r)) | m4)',
  '((! (((! ((! s) & (r | ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))) U s)) | m4)',
  '((((! m4) & (X m4)) V r) | m4)',
  '((((! m4) & (X m4)) V (! r)) | m4)',
  '(((! ((! m4) & (X m4))) U r) | m4)'],
 ['((((! m4) & (X m4)) V (c1 -> r)) | m4)',
  '((((! m4) & (X m4)) V (c1 -> (((! m4) & (X m4)) | ((X r) & (! ((! m4) & (X m4))))))) | m4)',
  '((((! m4) & (X m4)) V (c1 -> ((G[0,2] r) | (((! m4) & (X m4)) V r)))) | m4)',
  '((((! m4) & (X m4)) V (c1 -> (((G[0,2] (! r)) | (((! m4) & (X m4)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m4) & (X m4))))))) | m4)',
  '((((! m4) & (X m4)) V (c1 -> ((F[0,2] r) | (F[0,1] ((! m4) & (X m4)))))) | m4)',
  '((((! m4) & (X m4)) V (c1 -> ((! (((! s) & (! ((! m4) & (X m4)))) U ((! r) & (! s)))) | (((r & ((! m4) & (X m4))) | ((! m4) & (X m4))) V r)))) | m4)',
  '((((! m4) & (X m4)) V (c1 -> (! (((! ((! s) & (r | ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))) U s)))) | m4)',
  '((((! m4) & (X m4)) V (c1 -> (((! m4) & (X m4)) V r))) | m4)',
  '((((! m4) & (X m4)) V (c1 -> (((! m4) & (X m4)) V (! r)))) | m4)',
  '((((! m4) & (X m4)) V (c1 -> ((! ((! m4) & (X m4))) U r))) | m4)'],
 ['(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X r) & (! ((! m4) & (X m4)))))) & (c2 -> r)) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X (((! m4) & (X m4)) | ((X r) & (! ((! m4) & (X m4)))))) & (! ((! m4) & (X m4)))))) & (c2 -> (((! m4) & (X m4)) | ((X r) & (! ((! m4) & (X m4))))))) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X ((G[0,2] r) | (((! m4) & (X m4)) V r))) & (! ((! m4) & (X m4)))))) & (c2 -> ((G[0,2] r) | (((! m4) & (X m4)) V r)))) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X (((G[0,2] (! r)) | (((! m4) & (X m4)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m4) & (X m4)))))) & (! ((! m4) & (X m4)))))) & (c2 -> (((G[0,2] (! r)) | (((! m4) & (X m4)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m4) & (X m4))))))) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X ((F[0,2] r) | (F[0,1] ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))))) & (c2 -> ((F[0,2] r) | (F[0,1] ((! m4) & (X m4)))))) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X ((! (((! s) & (! ((! m4) & (X m4)))) U ((! r) & (! s)))) | (((r & ((! m4) & (X m4))) | ((! m4) & (X m4))) V r))) & (! ((! m4) & (X m4)))))) & (c2 -> ((! (((! s) & (! ((! m4) & (X m4)))) U ((! r) & (! s)))) | (((r & ((! m4) & (X m4))) | ((! m4) & (X m4))) V r)))) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X (! (((! ((! s) & (r | ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))) U s))) & (! ((! m4) & (X m4)))))) & (c2 -> (! (((! ((! s) & (r | ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))) U s)))) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X (((! m4) & (X m4)) V r)) & (! ((! m4) & (X m4)))))) & (c2 -> (((! m4) & (X m4)) V r))) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X (((! m4) & (X m4)) V (! r))) & (! ((! m4) & (X m4)))))) & (c2 -> (((! m4) & (X m4)) V (! r)))) | m4)',
  '(((((! m4) & (X m4)) V (((! c2) & ((X c2) & (! ((! m4) & (X m4))))) -> ((X ((! ((! m4) & (X m4))) U r)) & (! ((! m4) & (X m4)))))) & (c2 -> ((! ((! m4) & (X m4))) U r))) | m4)'],
 ['((G ((! (m5 & (X (! m5)))) | (X (! r)))) & ((! m5) -> (! r)))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))) & ((! m5) -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))) & ((! m5) -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))) & ((! m5) -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))) & ((! m5) -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))) & ((! m5) -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))) & ((! m5) -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((! ((! m5) & (X m5))) U (! r))))) & ((! m5) -> ((! ((! m5) & (X m5))) U (! r))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((! ((! m5) & (X m5))) U r)))) & ((! m5) -> ((! ((! m5) & (X m5))) U r)))',
  '(G (r -> m5))'],
 ['((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> (! r)))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> (! r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> ((! ((! m5) & (X m5))) U (! r))))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> ((! ((! m5) & (X m5))) U (! r))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> ((! ((! m5) & (X m5))) U r)))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> ((! ((! m5) & (X m5))) U r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c1 -> (((! m5) & (X m5)) V (! r))))))) & ((! m5) -> (((! m5) & (X m5)) V (c1 -> (((! m5) & (X m5)) V (! r))))))'],
 ['((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (! r)) & (! ((! m5) & (X m5)))))) & (c2 -> (! r)))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (! r)) & (! ((! m5) & (X m5)))))) & (c2 -> (! r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))) & (! ((! m5) & (X m5)))))) & (c2 -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))) & (! ((! m5) & (X m5)))))) & (c2 -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))))) & (c2 -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))))) & (c2 -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))) & (! ((! m5) & (X m5)))))) & (c2 -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))) & (! ((! m5) & (X m5)))))) & (c2 -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))) & (! ((! m5) & (X m5)))))) & (c2 -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))) & (! ((! m5) & (X m5)))))) & (c2 -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))) & (! ((! m5) & (X m5)))))) & (c2 -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))) & (! ((! m5) & (X m5)))))) & (c2 -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))) & (! ((! m5) & (X m5)))))) & (c2 -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))) & (! ((! m5) & (X m5)))))) & (c2 -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((! ((! m5) & (X m5))) U (! r))) & (! ((! m5) & (X m5)))))) & (c2 -> ((! ((! m5) & (X m5))) U (! r))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((! ((! m5) & (X m5))) U (! r))) & (! ((! m5) & (X m5)))))) & (c2 -> ((! ((! m5) & (X m5))) U (! r))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((! ((! m5) & (X m5))) U r)) & (! ((! m5) & (X m5)))))) & (c2 -> ((! ((! m5) & (X m5))) U r)))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X ((! ((! m5) & (X m5))) U r)) & (! ((! m5) & (X m5)))))) & (c2 -> ((! ((! m5) & (X m5))) U r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (((! m5) & (X m5)) V (! r))) & (! ((! m5) & (X m5)))))) & (c2 -> (((! m5) & (X m5)) V (! r))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c2) & ((X c2) & (! ((! m5) & (X m5))))) -> ((X (((! m5) & (X m5)) V (! r))) & (! ((! m5) & (X m5)))))) & (c2 -> (((! m5) & (X m5)) V (! r))))))'],
 ['(! r)',
  '((m6 & (X (! m6))) | ((X (! r)) & (! (m6 & (X (! m6))))))',
  '((F[0,2] (! r)) | (F[0,1] (m6 & (X (! m6)))))',
  '(((F[0,2] r) | (F[0,1] (m6 & (X (! m6))))) | ((G[0,3] (! r)) | ((m6 & (X (! m6))) V (! r))))',
  '((G[0,2] (! r)) | ((m6 & (X (! m6))) V (! r)))',
  '(! (((! ((! s) & ((! r) | (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))) U s))',
  '((! (((! s) & (! (m6 & (X (! m6))))) U (r & (! s)))) | (! (((! ((! r) & (m6 & (X (! m6))))) & (! (m6 & (X (! m6))))) U r)))',
  '((! (m6 & (X (! m6)))) U (! r))',
  '((! (m6 & (X (! m6)))) U r)',
  '((m6 & (X (! m6))) V (! r))'],
 ['((m6 & (X (! m6))) V (c1 -> (! r)))',
  '((m6 & (X (! m6))) V (c1 -> ((m6 & (X (! m6))) | ((X (! r)) & (! (m6 & (X (! m6))))))))',
  '((m6 & (X (! m6))) V (c1 -> ((F[0,2] (! r)) | (F[0,1] (m6 & (X (! m6)))))))',
  '((m6 & (X (! m6))) V (c1 -> (((F[0,2] r) | (F[0,1] (m6 & (X (! m6))))) | ((G[0,3] (! r)) | ((m6 & (X (! m6))) V (! r))))))',
  '((m6 & (X (! m6))) V (c1 -> ((G[0,2] (! r)) | ((m6 & (X (! m6))) V (! r)))))',
  '((m6 & (X (! m6))) V (c1 -> (! (((! ((! s) & ((! r) | (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))) U s))))',
  '((m6 & (X (! m6))) V (c1 -> ((! (((! s) & (! (m6 & (X (! m6))))) U (r & (! s)))) | (! (((! ((! r) & (m6 & (X (! m6))))) & (! (m6 & (X (! m6))))) U r)))))',
  '((m6 & (X (! m6))) V (c1 -> ((! (m6 & (X (! m6)))) U (! r))))',
  '((m6 & (X (! m6))) V (c1 -> ((! (m6 & (X (! m6)))) U r)))',
  '((m6 & (X (! m6))) V (c1 -> ((m6 & (X (! m6))) V (! r))))'],
 ['(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X (! r)) & (! (m6 & (X (! m6))))))) & (c2 -> (! r)))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X ((m6 & (X (! m6))) | ((X (! r)) & (! (m6 & (X (! m6))))))) & (! (m6 & (X (! m6))))))) & (c2 -> ((m6 & (X (! m6))) | ((X (! r)) & (! (m6 & (X (! m6))))))))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X ((F[0,2] (! r)) | (F[0,1] (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))))) & (c2 -> ((F[0,2] (! r)) | (F[0,1] (m6 & (X (! m6)))))))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X (((F[0,2] r) | (F[0,1] (m6 & (X (! m6))))) | ((G[0,3] (! r)) | ((m6 & (X (! m6))) V (! r))))) & (! (m6 & (X (! m6))))))) & (c2 -> (((F[0,2] r) | (F[0,1] (m6 & (X (! m6))))) | ((G[0,3] (! r)) | ((m6 & (X (! m6))) V (! r))))))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X ((G[0,2] (! r)) | ((m6 & (X (! m6))) V (! r)))) & (! (m6 & (X (! m6))))))) & (c2 -> ((G[0,2] (! r)) | ((m6 & (X (! m6))) V (! r)))))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X (! (((! ((! s) & ((! r) | (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))) U s))) & (! (m6 & (X (! m6))))))) & (c2 -> (! (((! ((! s) & ((! r) | (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))) U s))))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X ((! (((! s) & (! (m6 & (X (! m6))))) U (r & (! s)))) | (! (((! ((! r) & (m6 & (X (! m6))))) & (! (m6 & (X (! m6))))) U r)))) & (! (m6 & (X (! m6))))))) & (c2 -> ((! (((! s) & (! (m6 & (X (! m6))))) U (r & (! s)))) | (! (((! ((! r) & (m6 & (X (! m6))))) & (! (m6 & (X (! m6))))) U r)))))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X ((! (m6 & (X (! m6)))) U (! r))) & (! (m6 & (X (! m6))))))) & (c2 -> ((! (m6 & (X (! m6)))) U (! r))))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X ((! (m6 & (X (! m6)))) U r)) & (! (m6 & (X (! m6))))))) & (c2 -> ((! (m6 & (X (! m6)))) U r)))',
  '(((m6 & (X (! m6))) V (((! c2) & ((X c2) & (! (m6 & (X (! m6)))))) -> ((X ((m6 & (X (! m6))) V (! r))) & (! (m6 & (X (! m6))))))) & (c2 -> ((m6 & (X (! m6))) V (! r))))'],
 ['(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (! r)))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (! r)))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (X (! r))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (X (! r))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (F[0,2] (! r))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (F[0,2] (! r))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G[0,2] (! r)) | (G (! r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G[0,2] (! r)) | (G (! r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (! (r U s))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (! (r U s))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((s V (! (r & (! s)))) | (! (F r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((s V (! (r & (! s)))) | (! (F r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (F (! r))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (F (! r))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (F r)))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (F r)))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (! r))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (! r))))'],
 ['(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> (! r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> (! r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> (X (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> (X (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> (F[0,2] (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> (F[0,2] (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> ((G[0,2] (! r)) | (G (! r)))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> ((G[0,2] (! r)) | (G (! r)))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> (! (r U s))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> (! (r U s))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> ((s V (! (r & (! s)))) | (! (F r)))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> ((s V (! (r & (! s)))) | (! (F r)))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> (F (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> (F (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> (F r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> (F r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c1 -> (G (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c1 -> (G (! r))))))'],
 ['(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X (! r)))) & (c2 -> (! r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X (! r)))) & (c2 -> (! r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X (X (! r))))) & (c2 -> (X (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X (X (! r))))) & (c2 -> (X (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X (F[0,2] (! r))))) & (c2 -> (F[0,2] (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X (F[0,2] (! r))))) & (c2 -> (F[0,2] (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))) & (c2 -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))) & (c2 -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X ((G[0,2] (! r)) | (G (! r)))))) & (c2 -> ((G[0,2] (! r)) | (G (! r)))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X ((G[0,2] (! r)) | (G (! r)))))) & (c2 -> ((G[0,2] (! r)) | (G (! r)))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X (! (r U s))))) & (c2 -> (! (r U s))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X (! (r U s))))) & (c2 -> (! (r U s))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X ((s V (! (r & (! s)))) | (! (F r)))))) & (c2 -> ((s V (! (r & (! s)))) | (! (F r)))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X ((s V (! (r & (! s)))) | (! (F r)))))) & (c2 -> ((s V (! (r & (! s)))) | (! (F r)))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X (F (! r))))) & (c2 -> (F (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X (F (! r))))) & (c2 -> (F (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X (F r)))) & (c2 -> (F r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X (F r)))) & (c2 -> (F r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c2) & (X c2)) -> (X (G (! r))))) & (c2 -> (G (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c2) & (X c2)) -> (X (G (! r))))) & (c2 -> (G (! r))))))']]

universal_var_names = ['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'c1', 'c2']


## fretish condition-timing templates

In [ ]:
all_f_list = [['r',
  '(X r)',
  '((G[0,2] r) | (G r))',
  '(((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))',
  '(F[0,2] r)',
  '(s V (r | s))',
  '(r V (! s))',
  '(G r)',
  '(G (! r))',
  '(F r)',
  '(G (c -> r))',
  '(G (c -> (X r)))',
  '(G (c -> ((G[0,2] r) | (G r))))',
  '(G (c -> (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))',
  '(G (c -> (F[0,2] r)))',
  '(G (c -> (s V (r | s))))',
  '(G (c -> (r V (! s))))',
  '(G (c -> (G r)))',
  '(G (c -> (G (! r))))',
  '(G (c -> (F r)))',
  '((G (((! c) & (X c)) -> (X r))) & (c -> r))',
  '((G (((! c) & (X c)) -> (X (X r)))) & (c -> (X r)))',
  '((G (((! c) & (X c)) -> (X ((G[0,2] r) | (G r))))) & (c -> ((G[0,2] r) | (G r))))',
  '((G (((! c) & (X c)) -> (X (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))) & (c -> (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))',
  '((G (((! c) & (X c)) -> (X (F[0,2] r)))) & (c -> (F[0,2] r)))',
  '((G (((! c) & (X c)) -> (X (s V (r | s))))) & (c -> (s V (r | s))))',
  '((G (((! c) & (X c)) -> (X (r V (! s))))) & (c -> (r V (! s))))',
  '((G (((! c) & (X c)) -> (X (G r)))) & (c -> (G r)))',
  '((G (((! c) & (X c)) -> (X (G (! r))))) & (c -> (G (! r))))',
  '((G (((! c) & (X c)) -> (X (F r)))) & (c -> (F r)))'],
 ['((G ((! ((! m1) & (X m1))) | (X r))) & (m1 -> r))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))) & (m1 -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((G[0,2] r) | ((m1 & (X (! m1))) V r))))) & (m1 -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))',
  '((G ((! ((! m1) & (X m1))) | (X (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))) & (m1 -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))) & (m1 -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))) & (m1 -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))',
  '((G ((! ((! m1) & (X m1))) | (X (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))) & (m1 -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))',
  '(G (m1 -> r))',
  '(G (m1 -> (! r)))',
  '((G ((! ((! m1) & (X m1))) | (X ((! (m1 & (X (! m1)))) U r)))) & (m1 -> ((! (m1 & (X (! m1)))) U r)))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> r))))) & (m1 -> ((m1 & (X (! m1))) V (c -> r))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))))) & (m1 -> ((m1 & (X (! m1))) V (c -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))))) & (m1 -> ((m1 & (X (! m1))) V (c -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))))) & (m1 -> ((m1 & (X (! m1))) V (c -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))))) & (m1 -> ((m1 & (X (! m1))) V (c -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))))) & (m1 -> ((m1 & (X (! m1))) V (c -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))))) & (m1 -> ((m1 & (X (! m1))) V (c -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> ((m1 & (X (! m1))) V r)))))) & (m1 -> ((m1 & (X (! m1))) V (c -> ((m1 & (X (! m1))) V r)))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> ((m1 & (X (! m1))) V (! r))))))) & (m1 -> ((m1 & (X (! m1))) V (c -> ((m1 & (X (! m1))) V (! r))))))',
  '((G ((! ((! m1) & (X m1))) | (X ((m1 & (X (! m1))) V (c -> ((! (m1 & (X (! m1)))) U r)))))) & (m1 -> ((m1 & (X (! m1))) V (c -> ((! (m1 & (X (! m1)))) U r)))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X r) & (! (m1 & (X (! m1))))))) & (c -> r))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X r) & (! (m1 & (X (! m1))))))) & (c -> r))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))) & (! (m1 & (X (! m1))))))) & (c -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))) & (! (m1 & (X (! m1))))))) & (c -> ((m1 & (X (! m1))) | ((X r) & (! (m1 & (X (! m1))))))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((G[0,2] r) | ((m1 & (X (! m1))) V r))) & (! (m1 & (X (! m1))))))) & (c -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((G[0,2] r) | ((m1 & (X (! m1))) V r))) & (! (m1 & (X (! m1))))))) & (c -> ((G[0,2] r) | ((m1 & (X (! m1))) V r))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))) & (! (m1 & (X (! m1))))))) & (c -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))) & (! (m1 & (X (! m1))))))) & (c -> (((G[0,2] (! r)) | ((m1 & (X (! m1))) V (! r))) & ((F[0,3] r) | (F[0,2] (m1 & (X (! m1))))))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))))) & (c -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))))) & (c -> ((F[0,2] r) | (F[0,1] (m1 & (X (! m1)))))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))) & (! (m1 & (X (! m1))))))) & (c -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))) & (! (m1 & (X (! m1))))))) & (c -> ((! (((! s) & (! (m1 & (X (! m1))))) U ((! r) & (! s)))) | (((r & (m1 & (X (! m1)))) | (m1 & (X (! m1)))) V r))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))) & (! (m1 & (X (! m1))))))) & (c -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))) & (! (m1 & (X (! m1))))))) & (c -> (! (((! ((! s) & (r | (m1 & (X (! m1)))))) & (! (m1 & (X (! m1))))) U s))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) V r)) & (! (m1 & (X (! m1))))))) & (c -> ((m1 & (X (! m1))) V r)))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) V r)) & (! (m1 & (X (! m1))))))) & (c -> ((m1 & (X (! m1))) V r)))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) V (! r))) & (! (m1 & (X (! m1))))))) & (c -> ((m1 & (X (! m1))) V (! r))))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((m1 & (X (! m1))) V (! r))) & (! (m1 & (X (! m1))))))) & (c -> ((m1 & (X (! m1))) V (! r))))))',
  '((G ((! ((! m1) & (X m1))) | (X (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((! (m1 & (X (! m1)))) U r)) & (! (m1 & (X (! m1))))))) & (c -> ((! (m1 & (X (! m1)))) U r)))))) & (m1 -> (((m1 & (X (! m1))) V (((! c) & ((X c) & (! (m1 & (X (! m1)))))) -> ((X ((! (m1 & (X (! m1)))) U r)) & (! (m1 & (X (! m1))))))) & (c -> ((! (m1 & (X (! m1)))) U r)))))'],
 ['((G ((! (m2 & (X (! m2)))) | (X r))) & ((! m2) -> r))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))) & ((! m2) -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((G[0,2] r) | (((! m2) & (X m2)) V r))))) & ((! m2) -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))) & ((! m2) -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))) & ((! m2) -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))) & ((! m2) -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))',
  '((G ((! (m2 & (X (! m2)))) | (X (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))) & ((! m2) -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))',
  '(G ((! m2) -> r))',
  '(G (r -> m2))',
  '((G ((! (m2 & (X (! m2)))) | (X ((! ((! m2) & (X m2))) U r)))) & ((! m2) -> ((! ((! m2) & (X m2))) U r)))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> r))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> r))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> (((! m2) & (X m2)) V r)))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> (((! m2) & (X m2)) V r)))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> (((! m2) & (X m2)) V (! r))))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> (((! m2) & (X m2)) V (! r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X (((! m2) & (X m2)) V (c -> ((! ((! m2) & (X m2))) U r)))))) & ((! m2) -> (((! m2) & (X m2)) V (c -> ((! ((! m2) & (X m2))) U r)))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X r) & (! ((! m2) & (X m2)))))) & (c -> r))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X r) & (! ((! m2) & (X m2)))))) & (c -> r))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))) & (! ((! m2) & (X m2)))))) & (c -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))) & (! ((! m2) & (X m2)))))) & (c -> (((! m2) & (X m2)) | ((X r) & (! ((! m2) & (X m2)))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X ((G[0,2] r) | (((! m2) & (X m2)) V r))) & (! ((! m2) & (X m2)))))) & (c -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X ((G[0,2] r) | (((! m2) & (X m2)) V r))) & (! ((! m2) & (X m2)))))) & (c -> ((G[0,2] r) | (((! m2) & (X m2)) V r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))) & (! ((! m2) & (X m2)))))) & (c -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))) & (! ((! m2) & (X m2)))))) & (c -> (((G[0,2] (! r)) | (((! m2) & (X m2)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m2) & (X m2)))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))))) & (c -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))))) & (c -> ((F[0,2] r) | (F[0,1] ((! m2) & (X m2))))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))) & (! ((! m2) & (X m2)))))) & (c -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))) & (! ((! m2) & (X m2)))))) & (c -> ((! (((! s) & (! ((! m2) & (X m2)))) U ((! r) & (! s)))) | (((r & ((! m2) & (X m2))) | ((! m2) & (X m2))) V r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))) & (! ((! m2) & (X m2)))))) & (c -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))) & (! ((! m2) & (X m2)))))) & (c -> (! (((! ((! s) & (r | ((! m2) & (X m2))))) & (! ((! m2) & (X m2)))) U s))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) V r)) & (! ((! m2) & (X m2)))))) & (c -> (((! m2) & (X m2)) V r)))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) V r)) & (! ((! m2) & (X m2)))))) & (c -> (((! m2) & (X m2)) V r)))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) V (! r))) & (! ((! m2) & (X m2)))))) & (c -> (((! m2) & (X m2)) V (! r))))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X (((! m2) & (X m2)) V (! r))) & (! ((! m2) & (X m2)))))) & (c -> (((! m2) & (X m2)) V (! r))))))',
  '((G ((! (m2 & (X (! m2)))) | (X ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X ((! ((! m2) & (X m2))) U r)) & (! ((! m2) & (X m2)))))) & (c -> ((! ((! m2) & (X m2))) U r)))))) & ((! m2) -> ((((! m2) & (X m2)) V (((! c) & ((X c) & (! ((! m2) & (X m2))))) -> ((X ((! ((! m2) & (X m2))) U r)) & (! ((! m2) & (X m2)))))) & (c -> ((! ((! m2) & (X m2))) U r)))))'],
 ['(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X r))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (X r)))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G[0,2] r) | (G r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (F[0,2] r)))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (s V (r | s))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (r V (! s))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G r)))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (! r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (F r)))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> (X r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> ((G[0,2] r) | (G r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> (F[0,2] r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> (s V (r | s))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> (r V (! s))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> (G r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> (G (! r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X (G (c -> (F r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X r))) & (c -> r))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X (X r)))) & (c -> (X r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X ((G[0,2] r) | (G r))))) & (c -> ((G[0,2] r) | (G r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))) & (c -> (((G[0,2] (! r)) | (G (! r))) & (F[0,3] r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X (F[0,2] r)))) & (c -> (F[0,2] r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X (s V (r | s))))) & (c -> (s V (r | s))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X (r V (! s))))) & (c -> (r V (! s))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X (G r)))) & (c -> (G r)))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X (G (! r))))) & (c -> (G (! r))))))) | (G (! (m3 & (X (! m3))))))',
  '(((! (m3 & (X (! m3)))) U ((m3 & (X (! m3))) & (X ((G (((! c) & (X c)) -> (X (F r)))) & (c -> (F r)))))) | (G (! (m3 & (X (! m3))))))'],
 ['(r | m4)',
  '((((! m4) & (X m4)) | ((X r) & (! ((! m4) & (X m4))))) | m4)',
  '(((G[0,2] r) | (((! m4) & (X m4)) V r)) | m4)',
  '((((G[0,2] (! r)) | (((! m4) & (X m4)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m4) & (X m4))))) | m4)',
  '(((F[0,2] r) | (F[0,1] ((! m4) & (X m4)))) | m4)',
  '(((! (((! s) & (! ((! m4) & (X m4)))) U ((! r) & (! s)))) | (((r & ((! m4) & (X m4))) | ((! m4) & (X m4))) V r)) | m4)',
  '((! (((! ((! s) & (r | ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))) U s)) | m4)',
  '((((! m4) & (X m4)) V r) | m4)',
  '((((! m4) & (X m4)) V (! r)) | m4)',
  '(((! ((! m4) & (X m4))) U r) | m4)',
  '((((! m4) & (X m4)) V (c -> r)) | m4)',
  '((((! m4) & (X m4)) V (c -> (((! m4) & (X m4)) | ((X r) & (! ((! m4) & (X m4))))))) | m4)',
  '((((! m4) & (X m4)) V (c -> ((G[0,2] r) | (((! m4) & (X m4)) V r)))) | m4)',
  '((((! m4) & (X m4)) V (c -> (((G[0,2] (! r)) | (((! m4) & (X m4)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m4) & (X m4))))))) | m4)',
  '((((! m4) & (X m4)) V (c -> ((F[0,2] r) | (F[0,1] ((! m4) & (X m4)))))) | m4)',
  '((((! m4) & (X m4)) V (c -> ((! (((! s) & (! ((! m4) & (X m4)))) U ((! r) & (! s)))) | (((r & ((! m4) & (X m4))) | ((! m4) & (X m4))) V r)))) | m4)',
  '((((! m4) & (X m4)) V (c -> (! (((! ((! s) & (r | ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))) U s)))) | m4)',
  '((((! m4) & (X m4)) V (c -> (((! m4) & (X m4)) V r))) | m4)',
  '((((! m4) & (X m4)) V (c -> (((! m4) & (X m4)) V (! r)))) | m4)',
  '((((! m4) & (X m4)) V (c -> ((! ((! m4) & (X m4))) U r))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X r) & (! ((! m4) & (X m4)))))) & (c -> r)) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X (((! m4) & (X m4)) | ((X r) & (! ((! m4) & (X m4)))))) & (! ((! m4) & (X m4)))))) & (c -> (((! m4) & (X m4)) | ((X r) & (! ((! m4) & (X m4))))))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X ((G[0,2] r) | (((! m4) & (X m4)) V r))) & (! ((! m4) & (X m4)))))) & (c -> ((G[0,2] r) | (((! m4) & (X m4)) V r)))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X (((G[0,2] (! r)) | (((! m4) & (X m4)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m4) & (X m4)))))) & (! ((! m4) & (X m4)))))) & (c -> (((G[0,2] (! r)) | (((! m4) & (X m4)) V (! r))) & ((F[0,3] r) | (F[0,2] ((! m4) & (X m4))))))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X ((F[0,2] r) | (F[0,1] ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))))) & (c -> ((F[0,2] r) | (F[0,1] ((! m4) & (X m4)))))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X ((! (((! s) & (! ((! m4) & (X m4)))) U ((! r) & (! s)))) | (((r & ((! m4) & (X m4))) | ((! m4) & (X m4))) V r))) & (! ((! m4) & (X m4)))))) & (c -> ((! (((! s) & (! ((! m4) & (X m4)))) U ((! r) & (! s)))) | (((r & ((! m4) & (X m4))) | ((! m4) & (X m4))) V r)))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X (! (((! ((! s) & (r | ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))) U s))) & (! ((! m4) & (X m4)))))) & (c -> (! (((! ((! s) & (r | ((! m4) & (X m4))))) & (! ((! m4) & (X m4)))) U s)))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X (((! m4) & (X m4)) V r)) & (! ((! m4) & (X m4)))))) & (c -> (((! m4) & (X m4)) V r))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X (((! m4) & (X m4)) V (! r))) & (! ((! m4) & (X m4)))))) & (c -> (((! m4) & (X m4)) V (! r)))) | m4)',
  '(((((! m4) & (X m4)) V (((! c) & ((X c) & (! ((! m4) & (X m4))))) -> ((X ((! ((! m4) & (X m4))) U r)) & (! ((! m4) & (X m4)))))) & (c -> ((! ((! m4) & (X m4))) U r))) | m4)'],
 ['((G ((! (m5 & (X (! m5)))) | (X (! r)))) & ((! m5) -> (! r)))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))) & ((! m5) -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))) & ((! m5) -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))) & ((! m5) -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))) & ((! m5) -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))) & ((! m5) -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))) & ((! m5) -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((! ((! m5) & (X m5))) U (! r))))) & ((! m5) -> ((! ((! m5) & (X m5))) U (! r))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((! ((! m5) & (X m5))) U r)))) & ((! m5) -> ((! ((! m5) & (X m5))) U r)))',
  '(G (r -> m5))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> (! r)))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> (! r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> ((! ((! m5) & (X m5))) U (! r))))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> ((! ((! m5) & (X m5))) U (! r))))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> ((! ((! m5) & (X m5))) U r)))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> ((! ((! m5) & (X m5))) U r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X (((! m5) & (X m5)) V (c -> (((! m5) & (X m5)) V (! r))))))) & ((! m5) -> (((! m5) & (X m5)) V (c -> (((! m5) & (X m5)) V (! r))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (! r)) & (! ((! m5) & (X m5)))))) & (c -> (! r)))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (! r)) & (! ((! m5) & (X m5)))))) & (c -> (! r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))) & (! ((! m5) & (X m5)))))) & (c -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))) & (! ((! m5) & (X m5)))))) & (c -> (((! m5) & (X m5)) | ((X (! r)) & (! ((! m5) & (X m5)))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))))) & (c -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))))) & (c -> ((F[0,2] (! r)) | (F[0,1] ((! m5) & (X m5))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))) & (! ((! m5) & (X m5)))))) & (c -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))) & (! ((! m5) & (X m5)))))) & (c -> (((F[0,2] r) | (F[0,1] ((! m5) & (X m5)))) | ((G[0,3] (! r)) | (((! m5) & (X m5)) V (! r))))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))) & (! ((! m5) & (X m5)))))) & (c -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))) & (! ((! m5) & (X m5)))))) & (c -> ((G[0,2] (! r)) | (((! m5) & (X m5)) V (! r)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))) & (! ((! m5) & (X m5)))))) & (c -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))) & (! ((! m5) & (X m5)))))) & (c -> (! (((! ((! s) & ((! r) | ((! m5) & (X m5))))) & (! ((! m5) & (X m5)))) U s))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))) & (! ((! m5) & (X m5)))))) & (c -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))) & (! ((! m5) & (X m5)))))) & (c -> ((! (((! s) & (! ((! m5) & (X m5)))) U (r & (! s)))) | (! (((! ((! r) & ((! m5) & (X m5)))) & (! ((! m5) & (X m5)))) U r)))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((! ((! m5) & (X m5))) U (! r))) & (! ((! m5) & (X m5)))))) & (c -> ((! ((! m5) & (X m5))) U (! r))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((! ((! m5) & (X m5))) U (! r))) & (! ((! m5) & (X m5)))))) & (c -> ((! ((! m5) & (X m5))) U (! r))))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((! ((! m5) & (X m5))) U r)) & (! ((! m5) & (X m5)))))) & (c -> ((! ((! m5) & (X m5))) U r)))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X ((! ((! m5) & (X m5))) U r)) & (! ((! m5) & (X m5)))))) & (c -> ((! ((! m5) & (X m5))) U r)))))',
  '((G ((! (m5 & (X (! m5)))) | (X ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (((! m5) & (X m5)) V (! r))) & (! ((! m5) & (X m5)))))) & (c -> (((! m5) & (X m5)) V (! r))))))) & ((! m5) -> ((((! m5) & (X m5)) V (((! c) & ((X c) & (! ((! m5) & (X m5))))) -> ((X (((! m5) & (X m5)) V (! r))) & (! ((! m5) & (X m5)))))) & (c -> (((! m5) & (X m5)) V (! r))))))'],
 ['(! r)',
  '((m6 & (X (! m6))) | ((X (! r)) & (! (m6 & (X (! m6))))))',
  '((F[0,2] (! r)) | (F[0,1] (m6 & (X (! m6)))))',
  '(((F[0,2] r) | (F[0,1] (m6 & (X (! m6))))) | ((G[0,3] (! r)) | ((m6 & (X (! m6))) V (! r))))',
  '((G[0,2] (! r)) | ((m6 & (X (! m6))) V (! r)))',
  '(! (((! ((! s) & ((! r) | (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))) U s))',
  '((! (((! s) & (! (m6 & (X (! m6))))) U (r & (! s)))) | (! (((! ((! r) & (m6 & (X (! m6))))) & (! (m6 & (X (! m6))))) U r)))',
  '((! (m6 & (X (! m6)))) U (! r))',
  '((! (m6 & (X (! m6)))) U r)',
  '((m6 & (X (! m6))) V (! r))',
  '((m6 & (X (! m6))) V (c -> (! r)))',
  '((m6 & (X (! m6))) V (c -> ((m6 & (X (! m6))) | ((X (! r)) & (! (m6 & (X (! m6))))))))',
  '((m6 & (X (! m6))) V (c -> ((F[0,2] (! r)) | (F[0,1] (m6 & (X (! m6)))))))',
  '((m6 & (X (! m6))) V (c -> (((F[0,2] r) | (F[0,1] (m6 & (X (! m6))))) | ((G[0,3] (! r)) | ((m6 & (X (! m6))) V (! r))))))',
  '((m6 & (X (! m6))) V (c -> ((G[0,2] (! r)) | ((m6 & (X (! m6))) V (! r)))))',
  '((m6 & (X (! m6))) V (c -> (! (((! ((! s) & ((! r) | (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))) U s))))',
  '((m6 & (X (! m6))) V (c -> ((! (((! s) & (! (m6 & (X (! m6))))) U (r & (! s)))) | (! (((! ((! r) & (m6 & (X (! m6))))) & (! (m6 & (X (! m6))))) U r)))))',
  '((m6 & (X (! m6))) V (c -> ((! (m6 & (X (! m6)))) U (! r))))',
  '((m6 & (X (! m6))) V (c -> ((! (m6 & (X (! m6)))) U r)))',
  '((m6 & (X (! m6))) V (c -> ((m6 & (X (! m6))) V (! r))))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X (! r)) & (! (m6 & (X (! m6))))))) & (c -> (! r)))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X ((m6 & (X (! m6))) | ((X (! r)) & (! (m6 & (X (! m6))))))) & (! (m6 & (X (! m6))))))) & (c -> ((m6 & (X (! m6))) | ((X (! r)) & (! (m6 & (X (! m6))))))))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X ((F[0,2] (! r)) | (F[0,1] (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))))) & (c -> ((F[0,2] (! r)) | (F[0,1] (m6 & (X (! m6)))))))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X (((F[0,2] r) | (F[0,1] (m6 & (X (! m6))))) | ((G[0,3] (! r)) | ((m6 & (X (! m6))) V (! r))))) & (! (m6 & (X (! m6))))))) & (c -> (((F[0,2] r) | (F[0,1] (m6 & (X (! m6))))) | ((G[0,3] (! r)) | ((m6 & (X (! m6))) V (! r))))))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X ((G[0,2] (! r)) | ((m6 & (X (! m6))) V (! r)))) & (! (m6 & (X (! m6))))))) & (c -> ((G[0,2] (! r)) | ((m6 & (X (! m6))) V (! r)))))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X (! (((! ((! s) & ((! r) | (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))) U s))) & (! (m6 & (X (! m6))))))) & (c -> (! (((! ((! s) & ((! r) | (m6 & (X (! m6)))))) & (! (m6 & (X (! m6))))) U s))))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X ((! (((! s) & (! (m6 & (X (! m6))))) U (r & (! s)))) | (! (((! ((! r) & (m6 & (X (! m6))))) & (! (m6 & (X (! m6))))) U r)))) & (! (m6 & (X (! m6))))))) & (c -> ((! (((! s) & (! (m6 & (X (! m6))))) U (r & (! s)))) | (! (((! ((! r) & (m6 & (X (! m6))))) & (! (m6 & (X (! m6))))) U r)))))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X ((! (m6 & (X (! m6)))) U (! r))) & (! (m6 & (X (! m6))))))) & (c -> ((! (m6 & (X (! m6)))) U (! r))))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X ((! (m6 & (X (! m6)))) U r)) & (! (m6 & (X (! m6))))))) & (c -> ((! (m6 & (X (! m6)))) U r)))',
  '(((m6 & (X (! m6))) V (((! c) & ((X c) & (! (m6 & (X (! m6)))))) -> ((X ((m6 & (X (! m6))) V (! r))) & (! (m6 & (X (! m6))))))) & (c -> ((m6 & (X (! m6))) V (! r))))'],
 ['(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (! r)))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (! r)))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (X (! r))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (X (! r))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (F[0,2] (! r))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (F[0,2] (! r))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G[0,2] (! r)) | (G (! r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G[0,2] (! r)) | (G (! r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (! (r U s))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (! (r U s))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((s V (! (r & (! s)))) | (! (F r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((s V (! (r & (! s)))) | (! (F r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (F (! r))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (F (! r))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (F r)))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (F r)))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (! r))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (! r))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> (! r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> (! r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> (X (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> (X (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> (F[0,2] (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> (F[0,2] (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> ((G[0,2] (! r)) | (G (! r)))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> ((G[0,2] (! r)) | (G (! r)))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> (! (r U s))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> (! (r U s))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> ((s V (! (r & (! s)))) | (! (F r)))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> ((s V (! (r & (! s)))) | (! (F r)))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> (F (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> (F (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> (F r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> (F r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X (G (c -> (G (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> (G (c -> (G (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X (! r)))) & (c -> (! r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X (! r)))) & (c -> (! r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X (X (! r))))) & (c -> (X (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X (X (! r))))) & (c -> (X (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X (F[0,2] (! r))))) & (c -> (F[0,2] (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X (F[0,2] (! r))))) & (c -> (F[0,2] (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))) & (c -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))) & (c -> ((F[0,2] r) | ((G[0,3] (! r)) | (G (! r))))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X ((G[0,2] (! r)) | (G (! r)))))) & (c -> ((G[0,2] (! r)) | (G (! r)))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X ((G[0,2] (! r)) | (G (! r)))))) & (c -> ((G[0,2] (! r)) | (G (! r)))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X (! (r U s))))) & (c -> (! (r U s))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X (! (r U s))))) & (c -> (! (r U s))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X ((s V (! (r & (! s)))) | (! (F r)))))) & (c -> ((s V (! (r & (! s)))) | (! (F r)))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X ((s V (! (r & (! s)))) | (! (F r)))))) & (c -> ((s V (! (r & (! s)))) | (! (F r)))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X (F (! r))))) & (c -> (F (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X (F (! r))))) & (c -> (F (! r))))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X (F r)))) & (c -> (F r)))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X (F r)))) & (c -> (F r)))))',
  '(((! m7) -> (((! ((! m7) & (X m7))) U (((! m7) & (X m7)) & (X ((G (((! c) & (X c)) -> (X (G (! r))))) & (c -> (G (! r))))))) | (G (! ((! m7) & (X m7)))))) & (m7 -> ((G (((! c) & (X c)) -> (X (G (! r))))) & (c -> (G (! r))))))']]

universal_var_names = ['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7']


## PSP templates

In [ ]:
#PSP

all_f_list = \
[['(G(!(bool_exp3)))',
  '((!(bool_exp3)) & (X((!(bool_exp3)) & (X(!(bool_exp3))))))',
  '(X(G(!(bool_exp3))))',
  '(X((!(bool_exp3)) & (X(!(bool_exp3)))))',
  '(G(bool_exp3))',
  '((bool_exp3) & (X((bool_exp3) & (X(bool_exp3)))))',
  '(X(G(bool_exp3)))',
  '(X((bool_exp3) & (X(bool_exp3))))',
  '(F(bool_exp3))',
  '((bool_exp3) | (X((bool_exp3) | (X(bool_exp3)))))',
  '(X(F(bool_exp3)))',
  '(X((bool_exp3) | (X(bool_exp3))))',
  '(G((bool_exp3) | ((!(bool_exp3)) U (((bool_exp3) & (X((bool_exp3) & (X(bool_exp3))))) | (G(!(bool_exp3)))))))',
  '(G((bool_exp3) | ((!(bool_exp3)) U (((bool_exp3) & ((!(bool_exp3)) | (X((!(bool_exp3)) | (X(!(bool_exp3))))))) | (G(!(bool_exp3)))))))',
  '(G((bool_exp3) | (X((bool_exp3) | (X(bool_exp3))))))',
  '(G((X(X(bool_exp3))) -> ((bool_exp4) | (X(bool_exp4)))))',
  '((!(bool_exp3)) U (bool_exp4))',
  '((bool_exp3) U (bool_exp4))',
  '((bool_exp4) | ((bool_exp3) & (X(bool_exp4))) | ((bool_exp3) & (X(bool_exp3)) & (X(X(bool_exp4)))))',
  '((bool_exp3) & (X(bool_exp3)) & (X((bool_exp3) U (bool_exp4))))',
  '(((bool_exp3) & (X(bool_exp4))) | ((bool_exp3) & (X(bool_exp3)) & (X(X(bool_exp4)))))',
  '(G((bool_exp3) -> ((TRUE) U (bool_exp4))))',
  '(G((bool_exp3) -> ((!(bool_cnt_exp3)) U (bool_exp4))))',
  '(G((bool_exp3) -> ((bool_exp4) | (X(bool_exp4)) | (X(X(bool_exp4))))))',
  '(G((bool_exp3) -> ((bool_exp4) | ((!(bool_cnt_exp3)) & (X(bool_exp4))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X(bool_exp4)))))))',
  '(G((bool_exp3) -> (X((TRUE) U (bool_exp4)))))',
  '(G((bool_exp3) -> ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X((!(bool_cnt_exp3)) U (bool_exp4))))))',
  '(G((bool_exp3) -> ((X(bool_exp4)) | (X(X(bool_exp4))))))',
  '(G((bool_exp3) -> (((!(bool_cnt_exp3)) & (X(bool_exp4))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X(bool_exp4)))))))',
  '(G((bool_exp3) -> (G(bool_exp4))))',
  '(G((bool_exp3) -> ((bool_exp4) & (X((bool_exp4) & (X(bool_exp4)))))))',
  '(G((bool_exp3) -> (X(G(bool_exp4)))))',
  '(G((bool_exp3) -> (X((bool_exp4) & (X(bool_exp4))))))',
  '(G((X(X((bool_exp4) & (X((bool_exp5) | (X((bool_exp5) | (X(bool_exp5))))))))) -> ((bool_exp3) | (X(bool_exp3)))))',
  '((F((bool_exp4) & (X(F(bool_exp5))))) -> ((!(bool_exp4)) U (bool_exp3)))',
  '(G((X(X(bool_exp3))) -> (((bool_exp4) & (X(bool_exp5))) | (X((bool_exp4) & (X(bool_exp5)))))))',
  '((F(bool_exp3)) -> ((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5)))))))',
  '(G((bool_exp3) -> (((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (X((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))))',
  '(G((bool_exp3) -> (((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | (X((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))) | (X(X((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))))))',
  '(G((bool_exp3) -> ((!(bool_cnt_exp3)) U ((!(bool_cnt_exp4)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))))',
  '(G((bool_exp3) -> ((TRUE) U ((bool_exp4) & (X((TRUE) U (bool_exp5)))))))',
  '(G((bool_exp3) -> (((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (X((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))))',
  '(G((bool_exp3) -> (((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | (X((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))) | (X(X((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))))))',
  '(G((bool_exp3) -> ((!(bool_cnt_exp3)) U ((!(bool_cnt_exp4)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))))',
  '(G((bool_exp3) -> ((TRUE) U ((bool_exp4) & (X((TRUE) U (bool_exp5)))))))',
  '(G((X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))))))))))) -> ((bool_exp3) | (X(bool_exp3)))))',
  '((F((bool_exp4) & (X(F((bool_exp5) & (X(F(bool_exp6)))))))) -> ((!(bool_exp4)) U (bool_exp3)))',
  '(G((X(X(bool_exp3))) -> (((bool_exp4) & (X((bool_exp5) & (X(bool_exp6))))) | (X((bool_exp4) & (X((bool_exp5) & (X(bool_exp6)))))))))',
  '((F(bool_exp3)) -> ((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp6))))))))))',
  '(G((bool_exp3) -> (((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (X((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))))',
  '(G((bool_exp3) -> (((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | (X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))) | (X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))))))',
  '(G((bool_exp3) -> ((!(bool_cnt_exp3)) U ((!(bool_cnt_exp4)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))))',
  '(G((bool_exp3) -> ((TRUE) U ((bool_exp4) & (X((TRUE) U ((bool_exp5) & (X((TRUE) U (bool_exp6))))))))))',
  '(G((bool_exp3) -> (((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (X((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))))',
  '(G((bool_exp3) -> (((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | (X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))) | (X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))))))',
  '(G((bool_exp3) -> ((!(bool_cnt_exp3)) U ((!(bool_cnt_exp4)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))))',
  '(G((bool_exp3) -> ((TRUE) U ((bool_exp4) & (X((TRUE) U ((bool_exp5) & (X((TRUE) U (bool_exp6))))))))))'],
 ['((F(r1)) -> (((!(bool_exp3)) U (r1)) | (G(!(bool_exp3)))))',
  '((F(r1)) -> ((r1) | ((!(bool_exp3)) & (X(r1))) | ((!(bool_exp3)) & (X(!(bool_exp3))) & (X(X(r1)))) | ((!(bool_exp3)) & (X((!(bool_exp3)) & (X(!(bool_exp3))))))))',
  '((F(r1)) -> ((X((!(bool_exp3)) U (r1))) | (X(G(!(bool_exp3))))))',
  '((F(r1)) -> ((X((r1) | ((!(bool_exp3)) & (X(r1))))) | (X((!(bool_exp3)) & (X(!(bool_exp3)))))))',
  '((F(r1)) -> (((bool_exp3) U (r1)) | (G(bool_exp3))))',
  '((F(r1)) -> ((r1) | ((bool_exp3) & (X(r1))) | ((bool_exp3) & (X(bool_exp3)) & (X(X(r1)))) | ((bool_exp3) & (X((bool_exp3) & (X(bool_exp3)))))))',
  '((F(r1)) -> ((X((bool_exp3) U (r1))) | (X(G(bool_exp3)))))',
  '((F(r1)) -> ((X((r1) | ((bool_exp3) & (X(r1))))) | (X((bool_exp3) & (X(bool_exp3))))))',
  '(((!(r1)) U ((!(r1)) & (bool_exp3))) | (G(!(r1))))',
  '(((!(r1)) & (bool_exp3)) | ((!(r1)) & (X((!(r1)) & (bool_exp3)))) | ((!(r1)) & (X(!(r1))) & (X(X((!(r1)) & (bool_exp3))))) | (G(!(r1))))',
  '((G(!(r1))) | ((!(r1)) & (X(!(r1))) & (X((!(r1)) U ((!(r1)) & (bool_exp3))))))',
  '(((!(r1)) & (X((!(r1)) & (bool_exp3)))) | ((!(r1)) & (X(!(r1))) & (X(X((!(r1)) & (bool_exp3))))) | (G(!(r1))))',
  '((F(r1)) -> (((bool_exp3) | ((!(bool_exp3)) U ((r1) | ((!(r1)) & (bool_exp3) & (X((!(r1)) & (bool_exp3) & (X((!(r1)) & (bool_exp3))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) | ((!(bool_exp3)) U ((r1) | ((bool_exp3) & ((r1) | (!(bool_exp3)) | (X((r1) | (!(bool_exp3)) | (X((r1) | (!(bool_exp3))))))))))) U (r1)))',
  '((F(r1)) -> (((r1) | (bool_exp3) | (X((r1) | (bool_exp3) | (X((r1) | (bool_exp3)))))) U (r1)))',
  '((F(r1)) -> (((X(X(bool_exp3))) -> ((r1) | (bool_exp4) | (X(bool_exp4)) | (X((r1) | (X(r1)))))) U (r1)))',
  '((F(r1)) -> ((!(bool_exp3)) U ((r1) | (bool_exp4))))',
  '((F(r1)) -> (((!(r1)) & (bool_exp3)) U ((r1) | (bool_exp4))))',
  '((F(r1)) -> ((r1) | (bool_exp4) | ((!(r1)) & (bool_exp3) & (X((r1) | (bool_exp4)))) | ((!(r1)) & (bool_exp3) & (X((!(r1)) & (bool_exp3))) & (X(X((r1) | (bool_exp4)))))))',
  '((F(r1)) -> ((!(r1)) & (bool_exp3) & (X((!(r1)) & (bool_exp3))) & (X(((!(r1)) & (bool_exp3)) U ((r1) | (bool_exp4))))))',
  '((F(r1)) -> (((!(r1)) & (bool_exp3) & (X((r1) | (bool_exp4)))) | ((!(r1)) & (bool_exp3) & (X((!(r1)) & (bool_exp3))) & (X(X((r1) | (bool_exp4)))))))',
  '((F(r1)) -> (((bool_exp3) -> ((!(r1)) U ((!(r1)) & (bool_exp4)))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r1))) U ((!(r1)) & (bool_exp4)))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(r1)) & (bool_exp4)) | ((!(r1)) & (X((!(r1)) & (bool_exp4)))) | ((!(r1)) & (X(!(r1))) & (X(X((!(r1)) & (bool_exp4))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(r1)) & (bool_exp4)) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(r1)) & (bool_exp4)))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp3)) & (!(r1)))) & (X(X((!(r1)) & (bool_exp4))))))) U (r1)))',
  '((X(F(r1))) -> (((bool_exp3) -> ((!(r1)) & (X(!(r1))) & (X((!(r1)) U ((!(r1)) & (bool_exp4)))))) U (r1)))',
  '((X(F(r1))) -> (((bool_exp3) -> ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp3)) & (!(r1)))) & (X(((!(bool_cnt_exp3)) & (!(r1))) U ((!(r1)) & (bool_exp4)))))) U (r1)))',
  '((X(F(r1))) -> (((bool_exp3) -> (((!(r1)) & (X((!(r1)) & (bool_exp4)))) | ((!(r1)) & (X(!(r1))) & (X(X((!(r1)) & (bool_exp4))))))) U (r1)))',
  '((X(F(r1))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r1)) & (X((!(r1)) & (bool_exp4)))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp3)) & (!(r1)))) & (X(X((!(r1)) & (bool_exp4))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (G((!(r1)) & (bool_exp4)))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> ((!(r1)) & (bool_exp4) & (X((!(r1)) & (bool_exp4) & (X((!(r1)) & (bool_exp4))))))) U (r1)))',
  '((X(F(r1))) -> (((bool_exp3) -> (X(G((!(r1)) & (bool_exp4))))) U (r1)))',
  '((X(F(r1))) -> (((bool_exp3) -> (X((!(r1)) & (bool_exp4) & (X((!(r1)) & (bool_exp4)))))) U (r1)))',
  '((F(r1)) -> (((X(X((bool_exp4) & (X((bool_exp5) | (X((bool_exp5) | (X(bool_exp5))))))))) -> ((r1) | (bool_exp3) | (X(bool_exp3)) | (X((r1) | (X((r1) | (X((r1) | (X(r1)))))))))) U (r1)))',
  '((F(r1)) -> ((!((!(r1)) & (bool_exp4) & (X((!(r1)) U ((!(r1)) & (bool_exp5)))))) U ((r1) | (bool_exp3))))',
  '((F(r1)) -> (((r1) | ((X(X(bool_exp3))) -> (((bool_exp4) & (X(bool_exp5))) | (X((bool_exp4) & (X(bool_exp5)))))) | (X((r1) | (X(r1))))) U (r1)))',
  '((F(r1)) -> ((!(bool_exp3)) U ((r1) | ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5))))))))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp3)) & (!(r1)))) & (X(X((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(r1)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | ((!(r1)) & (X((!(r1)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))) | ((!(r1)) & (X(!(r1))) & (X(X((!(r1)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r1))) U ((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> ((!(r1)) U ((!(r1)) & (bool_exp4) & (X((TRUE) U (bool_exp5)))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp3)) & (!(r1)))) & (X(X((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(r1)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | ((!(r1)) & (X((!(r1)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))) | ((!(r1)) & (X(!(r1))) & (X(X((!(r1)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r1))) U ((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> ((!(r1)) U ((!(r1)) & (bool_exp4) & (X((TRUE) U (bool_exp5)))))) U (r1)))',
  '((F(r1)) -> (((X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))))))))))) -> ((r1) | (bool_exp3) | (X(bool_exp3)) | (X((r1) | (X((r1) | (X((r1) | (X((r1) | (X((r1) | (X(r1)))))))))))))) U (r1)))',
  '((F(r1)) -> ((!((!(r1)) & (bool_exp4) & (X((!(r1)) U ((!(r1)) & (bool_exp5) & (X((!(r1)) U ((!(r1)) & (bool_exp6))))))))) U ((r1) | (bool_exp3))))',
  '((F(r1)) -> (((r1) | ((X(X(bool_exp3))) -> (((bool_exp4) & (X((bool_exp5) & (X(bool_exp6))))) | (X((bool_exp4) & (X((bool_exp5) & (X(bool_exp6)))))))) | (X((r1) | (X(r1))))) U (r1)))',
  '((F(r1)) -> ((!(bool_exp3)) U ((r1) | ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp6)))))))))))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp3)) & (!(r1)))) & (X(X((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(r1)) & (bool_exp4) & (X(((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | ((!(r1)) & (X((!(r1)) & (bool_exp4) & (X(((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))) | ((!(r1)) & (X(!(r1))) & (X(X((!(r1)) & (bool_exp4) & (X(((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r1))) U ((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> ((!(r1)) U ((!(r1)) & (bool_exp4) & (X((TRUE) U ((!(r1)) & (bool_exp5) & (X((TRUE) U (bool_exp6))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (!(r1)) & (X((!(bool_cnt_exp3)) & (!(r1)))) & (X(X((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(r1)) & (bool_exp4) & (X(((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | ((!(r1)) & (X((!(r1)) & (bool_exp4) & (X(((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))) | ((!(r1)) & (X(!(r1))) & (X(X((!(r1)) & (bool_exp4) & (X(((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r1)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r1))) U ((!(bool_cnt_exp4)) & (!(r1)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (!(r1)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))) U (r1)))',
  '((F(r1)) -> (((bool_exp3) -> ((!(r1)) U ((!(r1)) & (bool_exp4) & (X((TRUE) U ((!(r1)) & (bool_exp5) & (X((TRUE) U (bool_exp6))))))))) U (r1)))'],
 ['(G((q1) -> (G(!(bool_exp3)))))',
  '(G((q1) -> ((!(bool_exp3)) & (X((!(bool_exp3)) & (X(!(bool_exp3))))))))',
  '(G((q1) -> (X(G(!(bool_exp3))))))',
  '(G((q1) -> (X((!(bool_exp3)) & (X(!(bool_exp3)))))))',
  '(G((q1) -> (G(bool_exp3))))',
  '(G((q1) -> ((bool_exp3) & (X((bool_exp3) & (X(bool_exp3)))))))',
  '(G((q1) -> (X(G(bool_exp3)))))',
  '(G((q1) -> (X((bool_exp3) & (X(bool_exp3))))))',
  '((G(!(q1))) | (F((q1) & (F(bool_exp3)))))',
  '((G(!(q1))) | (F((q1) & ((bool_exp3) | (X((bool_exp3) | (X(bool_exp3))))))))',
  '((G(!(q1))) | (F((q1) & (X(F(bool_exp3))))))',
  '((G(!(q1))) | (F((q1) & (X((bool_exp3) | (X(bool_exp3)))))))',
  '(G((q1) -> (G((bool_exp3) | ((!(bool_exp3)) U (((bool_exp3) & (X((bool_exp3) & (X(bool_exp3))))) | (G(!(bool_exp3)))))))))',
  '(G((q1) -> (G((bool_exp3) | ((!(bool_exp3)) U (((bool_exp3) & ((!(bool_exp3)) | (X((!(bool_exp3)) | (X(!(bool_exp3))))))) | (G(!(bool_exp3)))))))))',
  '(G((q1) -> (G((bool_exp3) | (X((bool_exp3) | (X(bool_exp3))))))))',
  '(G((q1) -> (G((X(X(bool_exp3))) -> ((bool_exp4) | (X(bool_exp4)))))))',
  '((G(!(q1))) | (F((q1) & ((!(bool_exp3)) U (bool_exp4)))))',
  '(G((q1) -> ((bool_exp3) U (bool_exp4))))',
  '(G((q1) -> ((bool_exp4) | ((bool_exp3) & (X(bool_exp4))) | ((bool_exp3) & (X(bool_exp3)) & (X(X(bool_exp4)))))))',
  '(G((q1) -> ((bool_exp3) & (X(bool_exp3)) & (X((bool_exp3) U (bool_exp4))))))',
  '(G((q1) -> (((bool_exp3) & (X(bool_exp4))) | ((bool_exp3) & (X(bool_exp3)) & (X(X(bool_exp4)))))))',
  '(G((q1) -> (G((bool_exp3) -> ((TRUE) U (bool_exp4))))))',
  '(G((q1) -> (G((bool_exp3) -> ((!(bool_cnt_exp3)) U (bool_exp4))))))',
  '(G((q1) -> (G((bool_exp3) -> ((bool_exp4) | (X(bool_exp4)) | (X(X(bool_exp4))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((bool_exp4) | ((!(bool_cnt_exp3)) & (X(bool_exp4))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X(bool_exp4)))))))))',
  '(G((q1) -> (G((bool_exp3) -> (X((TRUE) U (bool_exp4)))))))',
  '(G((q1) -> (G((bool_exp3) -> ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X((!(bool_cnt_exp3)) U (bool_exp4))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((X(bool_exp4)) | (X(X(bool_exp4))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((!(bool_cnt_exp3)) & (X(bool_exp4))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X(bool_exp4)))))))))',
  '(G((q1) -> (G((bool_exp3) -> (G(bool_exp4))))))',
  '(G((q1) -> (G((bool_exp3) -> ((bool_exp4) & (X((bool_exp4) & (X(bool_exp4)))))))))',
  '(G((q1) -> (G((bool_exp3) -> (X(G(bool_exp4)))))))',
  '(G((q1) -> (G((bool_exp3) -> (X((bool_exp4) & (X(bool_exp4))))))))',
  '(G((q1) -> (G((X(X((bool_exp4) & (X((bool_exp5) | (X((bool_exp5) | (X(bool_exp5))))))))) -> ((bool_exp3) | (X(bool_exp3)))))))',
  '((G(!(q1))) | ((!(q1)) U ((q1) & ((F((bool_exp4) & (X(F(bool_exp5))))) -> ((!(bool_exp4)) U (bool_exp3))))))',
  '(G((q1) -> (G((X(X(bool_exp3))) -> (((bool_exp4) & (X(bool_exp5))) | (X((bool_exp4) & (X(bool_exp5)))))))))',
  '((G(!(q1))) | ((!(q1)) U (((q1) & (F(bool_exp3))) -> ((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5)))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (X((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | (X((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))) | (X(X((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((!(bool_cnt_exp3)) U ((!(bool_cnt_exp4)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((TRUE) U ((bool_exp4) & (X((TRUE) U (bool_exp5)))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (X((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X((!(bool_cnt_exp4)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | (X((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))) | (X(X((bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((!(bool_cnt_exp3)) U ((!(bool_cnt_exp4)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((TRUE) U ((bool_exp4) & (X((TRUE) U (bool_exp5)))))))))',
  '(G((q1) -> (G((X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))))))))))) -> ((bool_exp3) | (X(bool_exp3)))))))',
  '((G(!(q1))) | ((!(q1)) U ((q1) & ((F((bool_exp4) & (X(F((bool_exp5) & (X(F(bool_exp6)))))))) -> ((!(bool_exp4)) U (bool_exp3))))))',
  '(G((q1) -> (G((X(X(bool_exp3))) -> (((bool_exp4) & (X((bool_exp5) & (X(bool_exp6))))) | (X((bool_exp4) & (X((bool_exp5) & (X(bool_exp6)))))))))))',
  '((G(!(q1))) | ((!(q1)) U (((q1) & (F(bool_exp3))) -> ((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp6))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (X((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | (X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))) | (X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((!(bool_cnt_exp3)) U ((!(bool_cnt_exp4)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((TRUE) U ((bool_exp4) & (X((TRUE) U ((bool_exp5) & (X((TRUE) U (bool_exp6))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (X((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (X(!(bool_cnt_exp3))) & (X(X((!(bool_cnt_exp4)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> (((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | (X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))) | (X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((!(bool_cnt_exp3)) U ((!(bool_cnt_exp4)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))))))',
  '(G((q1) -> (G((bool_exp3) -> ((TRUE) U ((bool_exp4) & (X((TRUE) U ((bool_exp5) & (X((TRUE) U (bool_exp6))))))))))))'],
 ['(G(((q2) & (!(r2)) & (F(r2))) -> (((!(bool_exp3)) U (r2)) | (G(!(bool_exp3))))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> ((r2) | ((!(bool_exp3)) & (X(r2))) | ((!(bool_exp3)) & (X(!(bool_exp3))) & (X(X(r2)))) | ((!(bool_exp3)) & (X((!(bool_exp3)) & (X(!(bool_exp3)))))))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> ((X((!(bool_exp3)) U (r2))) | (X(G(!(bool_exp3)))))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> ((X((r2) | ((!(bool_exp3)) & (X(r2))))) | (X((!(bool_exp3)) & (X(!(bool_exp3))))))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) U (r2)) | (G(bool_exp3)))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> ((r2) | ((bool_exp3) & (X(r2))) | ((bool_exp3) & (X(bool_exp3)) & (X(X(r2)))) | ((bool_exp3) & (X((bool_exp3) & (X(bool_exp3))))))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> ((X((bool_exp3) U (r2))) | (X(G(bool_exp3))))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> ((X((r2) | ((bool_exp3) & (X(r2))))) | (X((bool_exp3) & (X(bool_exp3)))))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> ((!(r2)) U ((!(r2)) & (bool_exp3)))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((!(r2)) & (bool_exp3)) | ((!(r2)) & (X((!(r2)) & (bool_exp3)))) | ((!(r2)) & (X(!(r2))) & (X(X((!(r2)) & (bool_exp3))))))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> ((!(r2)) & (X(!(r2))) & (X((!(r2)) U ((!(r2)) & (bool_exp3)))))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> (((!(r2)) & (X((!(r2)) & (bool_exp3)))) | ((!(r2)) & (X(!(r2))) & (X(X((!(r2)) & (bool_exp3))))))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) | ((!(bool_exp3)) U ((r2) | ((!(r2)) & (bool_exp3) & (X((!(r2)) & (bool_exp3) & (X((!(r2)) & (bool_exp3))))))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) | ((!(bool_exp3)) U ((r2) | ((bool_exp3) & ((r2) | (!(bool_exp3)) | (X((r2) | (!(bool_exp3)) | (X((r2) | (!(bool_exp3))))))))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((r2) | (bool_exp3) | (X((r2) | (bool_exp3) | (X((r2) | (bool_exp3)))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((X(X(bool_exp3))) -> ((r2) | (bool_exp4) | (X(bool_exp4)) | (X((r2) | (X(r2)))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> ((!(bool_exp3)) U ((r2) | (bool_exp4)))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((!(r2)) & (bool_exp3)) U ((r2) | (bool_exp4)))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> ((r2) | (bool_exp4) | ((!(r2)) & (bool_exp3) & (X((r2) | (bool_exp4)))) | ((!(r2)) & (bool_exp3) & (X((!(r2)) & (bool_exp3))) & (X(X((r2) | (bool_exp4))))))))',
  '(G(((q2) & (!(r2)) & (X(F(r2)))) -> ((!(r2)) & (bool_exp3) & (X((!(r2)) & (bool_exp3))) & (X(((!(r2)) & (bool_exp3)) U ((r2) | (bool_exp4)))))))',
  '(G(((q2) & (!(r2)) & (X(F(r2)))) -> (((!(r2)) & (bool_exp3) & (X((r2) | (bool_exp4)))) | ((!(r2)) & (bool_exp3) & (X((!(r2)) & (bool_exp3))) & (X(X((r2) | (bool_exp4))))))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) -> ((!(r2)) U ((!(r2)) & (bool_exp4)))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r2))) U ((!(r2)) & (bool_exp4)))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) -> (((!(r2)) & (bool_exp4)) | ((!(r2)) & (X((!(r2)) & (bool_exp4)))) | ((!(r2)) & (X(!(r2))) & (X(X((!(r2)) & (bool_exp4))))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) -> (((!(r2)) & (bool_exp4)) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(r2)) & (bool_exp4)))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp3)) & (!(r2)))) & (X(X((!(r2)) & (bool_exp4))))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> (((bool_exp3) -> ((!(r2)) & (X(!(r2))) & (X((!(r2)) U ((!(r2)) & (bool_exp4)))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> (((bool_exp3) -> ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp3)) & (!(r2)))) & (X(((!(bool_cnt_exp3)) & (!(r2))) U ((!(r2)) & (bool_exp4)))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> (((bool_exp3) -> (((!(r2)) & (X((!(r2)) & (bool_exp4)))) | ((!(r2)) & (X(!(r2))) & (X(X((!(r2)) & (bool_exp4))))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r2)) & (X((!(r2)) & (bool_exp4)))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp3)) & (!(r2)))) & (X(X((!(r2)) & (bool_exp4))))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) -> (G((!(r2)) & (bool_exp4)))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((bool_exp3) -> ((!(r2)) & (bool_exp4) & (X((!(r2)) & (bool_exp4) & (X((!(r2)) & (bool_exp4))))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> (((bool_exp3) -> (X(G((!(r2)) & (bool_exp4))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (X(!(r2))) & (X(F(r2)))) -> (((bool_exp3) -> (X((!(r2)) & (bool_exp4) & (X((!(r2)) & (bool_exp4)))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((X(X((bool_exp4) & (X((bool_exp5) | (X((bool_exp5) | (X(bool_exp5))))))))) -> ((r2) | (bool_exp3) | (X(bool_exp3)) | (X((r2) | (X((r2) | (X((r2) | (X(r2)))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> ((!((!(r2)) & (bool_exp4) & (X((!(r2)) U ((!(r2)) & (bool_exp5)))))) U ((r2) | (bool_exp3)))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((r2) | ((X(X(bool_exp3))) -> (((bool_exp4) & (X(bool_exp5))) | (X((bool_exp4) & (X(bool_exp5)))))) | (X((r2) | (X(r2))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> ((!(bool_exp3)) U ((r2) | ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5)))))))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp3)) & (!(r2)))) & (X(X((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(r2)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | ((!(r2)) & (X((!(r2)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))) | ((!(r2)) & (X(!(r2))) & (X(X((!(r2)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r2))) U ((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> ((!(r2)) U ((!(r2)) & (bool_exp4) & (X((TRUE) U (bool_exp5)))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp3)) & (!(r2)))) & (X(X((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(r2)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | ((!(r2)) & (X((!(r2)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))) | ((!(r2)) & (X(!(r2))) & (X(X((!(r2)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r2))) U ((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> ((!(r2)) U ((!(r2)) & (bool_exp4) & (X((TRUE) U (bool_exp5)))))) U (r2))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))))))))))) -> ((r2) | (bool_exp3) | (X(bool_exp3)) | (X((r2) | (X((r2) | (X((r2) | (X((r2) | (X((r2) | (X(r2)))))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> ((!((!(r2)) & (bool_exp4) & (X((!(r2)) U ((!(r2)) & (bool_exp5) & (X((!(r2)) U ((!(r2)) & (bool_exp6))))))))) U ((r2) | (bool_exp3)))))',
  '(G(((q2) & (!(r2)) & (F(r2))) -> (((r2) | ((X(X(bool_exp3))) -> (((bool_exp4) & (X((bool_exp5) & (X(bool_exp6))))) | (X((bool_exp4) & (X((bool_exp5) & (X(bool_exp6)))))))) | (X((r2) | (X(r2))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> ((!(bool_exp3)) U ((r2) | ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp6))))))))))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp3)) & (!(r2)))) & (X(X((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(r2)) & (bool_exp4) & (X(((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | ((!(r2)) & (X((!(r2)) & (bool_exp4) & (X(((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))) | ((!(r2)) & (X(!(r2))) & (X(X((!(r2)) & (bool_exp4) & (X(((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r2))) U ((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> ((!(r2)) U ((!(r2)) & (bool_exp4) & (X((TRUE) U ((!(r2)) & (bool_exp5) & (X((TRUE) U (bool_exp6))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (!(r2)) & (X((!(bool_cnt_exp3)) & (!(r2)))) & (X(X((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(r2)) & (bool_exp4) & (X(((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | ((!(r2)) & (X((!(r2)) & (bool_exp4) & (X(((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))) | ((!(r2)) & (X(!(r2))) & (X(X((!(r2)) & (bool_exp4) & (X(((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r2)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r2))) U ((!(bool_cnt_exp4)) & (!(r2)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (!(r2)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))) U (r2))))',
  '(G(((q2) & (F(r2))) -> (((bool_exp3) -> ((!(r2)) U ((!(r2)) & (bool_exp4) & (X((TRUE) U ((!(r2)) & (bool_exp5) & (X((TRUE) U (bool_exp6))))))))) U (r2))))'],
 ['(G(((q3) & (!(r3))) -> (((!(bool_exp3)) U (r3)) | (G(!(bool_exp3))))))',
  '(G(((q3) & (!(r3))) -> ((r3) | ((!(bool_exp3)) & (X(r3))) | ((!(bool_exp3)) & (X(!(bool_exp3))) & (X(X(r3)))) | ((!(bool_exp3)) & (X((!(bool_exp3)) & (X(!(bool_exp3)))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (X(((!(bool_exp3)) U (r3)) | (G(!(bool_exp3)))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (X((r3) | ((!(bool_exp3)) & (X(r3))) | ((!(bool_exp3)) & (X(!(bool_exp3))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) U (r3)) | (G(bool_exp3)))))',
  '(G(((q3) & (!(r3))) -> ((r3) | ((bool_exp3) & (X(r3))) | ((bool_exp3) & (X(bool_exp3)) & (X(X(r3)))) | ((bool_exp3) & (X((bool_exp3) & (X(bool_exp3))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (X(((bool_exp3) U (r3)) | (G(bool_exp3))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (X((r3) | ((bool_exp3) & (X(r3))) | ((bool_exp3) & (X(bool_exp3)))))))',
  '(G(((q3) & (!(r3))) -> ((!(r3)) U ((!(r3)) & (bool_exp3)))))',
  '(G(((q3) & (!(r3))) -> (((!(r3)) & (bool_exp3)) | ((!(r3)) & (X((!(r3)) & (bool_exp3)))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp3))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> ((!(r3)) & (X(!(r3))) & (X((!(r3)) U ((!(r3)) & (bool_exp3)))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (((!(r3)) & (X((!(r3)) & (bool_exp3)))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp3))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) | ((!(bool_exp3)) U ((r3) | ((!(r3)) & (bool_exp3) & (X((!(r3)) & (bool_exp3) & (X((!(r3)) & (bool_exp3)))))) | (G(!(bool_exp3)))))) U ((r3) | (G((bool_exp3) | ((!(bool_exp3)) U ((r3) | ((!(r3)) & (bool_exp3) & (X((!(r3)) & (bool_exp3) & (X((!(r3)) & (bool_exp3)))))) | (G(!(bool_exp3)))))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) | ((!(bool_exp3)) U ((r3) | ((bool_exp3) & ((r3) | (!(bool_exp3)) | (X((r3) | (!(bool_exp3)) | (X((r3) | (!(bool_exp3)))))))) | (G(!(bool_exp3)))))) U ((r3) | (G((bool_exp3) | ((!(bool_exp3)) U ((r3) | ((bool_exp3) & ((r3) | (!(bool_exp3)) | (X((r3) | (!(bool_exp3)) | (X((r3) | (!(bool_exp3)))))))) | (G(!(bool_exp3)))))))))))',
  '(G(((q3) & (!(r3))) -> (((r3) | (bool_exp3) | (X((r3) | (bool_exp3) | (X((r3) | (bool_exp3)))))) U ((r3) | (G((r3) | (bool_exp3) | (X((r3) | (bool_exp3) | (X((r3) | (bool_exp3)))))))))))',
  '(G(((q3) & (!(r3))) -> (((X(X(bool_exp3))) -> ((r3) | (bool_exp4) | (X(bool_exp4)) | (X((r3) | (X(r3)))))) U ((r3) | (G((X(X(bool_exp3))) -> ((r3) | (bool_exp4) | (X(bool_exp4)) | (X((r3) | (X(r3)))))))))))',
  '(G(((q3) & (!(r3))) -> ((!(bool_exp3)) U ((r3) | (bool_exp4)))))',
  '(G(((q3) & (!(r3))) -> ((bool_exp3) U ((r3) | (bool_exp4)))))',
  '(G(((q3) & (!(r3))) -> ((r3) | (bool_exp4) | ((bool_exp3) & (X((r3) | (bool_exp4)))) | ((bool_exp3) & (X(bool_exp3)) & (X(X((r3) | (bool_exp4))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> ((bool_exp3) & (X(bool_exp3)) & (X((bool_exp3) U ((r3) | (bool_exp4)))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (((bool_exp3) & (X((r3) | (bool_exp4)))) | ((bool_exp3) & (X(bool_exp3)) & (X(X((r3) | (bool_exp4))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4)))) U ((r3) | (G((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4)))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(r3)) & (bool_exp4)))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(r3)) & (bool_exp4)))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) -> (((!(r3)) & (bool_exp4)) | ((!(r3)) & (X((!(r3)) & (bool_exp4)))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4))))))) U ((r3) | (G((bool_exp3) -> (((!(r3)) & (bool_exp4)) | ((!(r3)) & (X((!(r3)) & (bool_exp4)))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4))))))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) -> (((!(r3)) & (bool_exp4)) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(r3)) & (bool_exp4)))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(r3)) & (bool_exp4))))))) U ((r3) | (G((bool_exp3) -> (((!(r3)) & (bool_exp4)) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(r3)) & (bool_exp4)))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(r3)) & (bool_exp4))))))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (((bool_exp3) -> ((!(r3)) & (X(!(r3))) & (X((!(r3)) U ((!(r3)) & (bool_exp4)))))) U ((r3) | (G((bool_exp3) -> ((!(r3)) & (X(!(r3))) & (X((!(r3)) U ((!(r3)) & (bool_exp4)))))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (((bool_exp3) -> ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(((!(bool_cnt_exp3)) & (!(r3))) U ((!(r3)) & (bool_exp4)))))) U ((r3) | (G((bool_exp3) -> ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(((!(bool_cnt_exp3)) & (!(r3))) U ((!(r3)) & (bool_exp4)))))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (((bool_exp3) -> (((!(r3)) & (X((!(r3)) & (bool_exp4)))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4))))))) U ((r3) | (G((bool_exp3) -> (((!(r3)) & (X((!(r3)) & (bool_exp4)))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4))))))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3)) & (X((!(r3)) & (bool_exp4)))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(r3)) & (bool_exp4))))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3)) & (X((!(r3)) & (bool_exp4)))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(r3)) & (bool_exp4))))))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) -> (G((!(r3)) & (bool_exp4)))) U ((r3) | (G((bool_exp3) -> (G((!(r3)) & (bool_exp4)))))))))',
  '(G(((q3) & (!(r3))) -> (((bool_exp3) -> ((!(r3)) & (bool_exp4) & (X((!(r3)) & (bool_exp4) & (X((!(r3)) & (bool_exp4))))))) U ((r3) | (G((bool_exp3) -> ((!(r3)) & (bool_exp4) & (X((!(r3)) & (bool_exp4) & (X((!(r3)) & (bool_exp4))))))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (((bool_exp3) -> (X(G((!(r3)) & (bool_exp4))))) U ((r3) | (G((bool_exp3) -> (X(G((!(r3)) & (bool_exp4))))))))))',
  '(G(((q3) & (!(r3)) & (X(!(r3)))) -> (((bool_exp3) -> (X((!(r3)) & (bool_exp4) & (X((!(r3)) & (bool_exp4)))))) U ((r3) | (G((bool_exp3) -> (X((!(r3)) & (bool_exp4) & (X((!(r3)) & (bool_exp4)))))))))))',
  '(G(((q3) & (!(r3))) -> (((X(X((bool_exp4) & (X((bool_exp5) | (X((bool_exp5) | (X(bool_exp5))))))))) -> ((r3) | (bool_exp3) | (X(bool_exp3)) | (X((r3) | (X((r3) | (X((r3) | (X(r3)))))))))) U ((r3) | (G((X(X((bool_exp4) & (X((bool_exp5) | (X((bool_exp5) | (X(bool_exp5))))))))) -> ((r3) | (bool_exp3) | (X(bool_exp3)) | (X((r3) | (X((r3) | (X((r3) | (X(r3)))))))))))))))',
  '(G((q3) -> (((!((!(r3)) & (bool_exp4) & (X((!(r3)) U ((!(r3)) & (bool_exp5)))))) U ((r3) | (bool_exp3))) | (G(!((bool_exp4) & (X(F(bool_exp5)))))))))',
  '(G(((q3) & (!(r3))) -> (((r3) | ((X(X(bool_exp3))) -> (((bool_exp4) & (X(bool_exp5))) | (X((bool_exp4) & (X(bool_exp5)))))) | (X((r3) | (X(r3))))) U ((r3) | (G((r3) | ((X(X(bool_exp3))) -> (((bool_exp4) & (X(bool_exp5))) | (X((bool_exp4) & (X(bool_exp5)))))) | (X((r3) | (X(r3))))))))))',
  '(G((q3) -> ((F(bool_exp3)) -> ((!(bool_exp3)) U ((r3) | ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | ((!(r3)) & (X((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))))))) U ((r3) | (G((bool_exp3) -> (((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | ((!(r3)) & (X((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))))))))',
  '(G((q3) -> (((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4) & (X((TRUE) U (bool_exp5)))))) U ((r3) | (G((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4) & (X((TRUE) U (bool_exp5)))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((bool_exp5) | ((!(bool_cnt_exp4)) & (X(bool_exp5))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X(bool_exp5)))))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | ((!(r3)) & (X((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))))))) U ((r3) | (G((bool_exp3) -> (((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))) | ((!(r3)) & (X((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5)))))))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4) & (X((bool_exp5) | (X(bool_exp5)) | (X(X(bool_exp5))))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U (bool_exp5)))))))))))',
  '(G((q3) -> (((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4) & (X((TRUE) U (bool_exp5)))))) U ((r3) | (G((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4) & (X((TRUE) U (bool_exp5)))))))))))',
  '(G(((q3) & (!(r3))) -> (((X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))))))))))) -> ((r3) | (bool_exp3) | (X(bool_exp3)) | (X((r3) | (X((r3) | (X((r3) | (X((r3) | (X((r3) | (X(r3)))))))))))))) U ((r3) | (G((X(X((bool_exp4) & (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X(((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))) | (X((bool_exp5) & (X((bool_exp6) | (X((bool_exp6) | (X(bool_exp6))))))))))))))) -> ((r3) | (bool_exp3) | (X(bool_exp3)) | (X((r3) | (X((r3) | (X((r3) | (X((r3) | (X((r3) | (X(r3)))))))))))))))))))',
  '(G((q3) -> (((!((!(r3)) & (bool_exp4) & (X((!(r3)) U ((!(r3)) & (bool_exp5) & (X((!(r3)) U ((!(r3)) & (bool_exp6))))))))) U ((r3) | (bool_exp3))) | (G(!((bool_exp4) & (X(F((bool_exp5) & (X(F(bool_exp6))))))))))))',
  '(G(((q3) & (!(r3))) -> (((r3) | ((X(X(bool_exp3))) -> (((bool_exp4) & (X((bool_exp5) & (X(bool_exp6))))) | (X((bool_exp4) & (X((bool_exp5) & (X(bool_exp6)))))))) | (X((r3) | (X(r3))))) U ((r3) | (G((r3) | ((X(X(bool_exp3))) -> (((bool_exp4) & (X((bool_exp5) & (X(bool_exp6))))) | (X((bool_exp4) & (X((bool_exp5) & (X(bool_exp6)))))))) | (X((r3) | (X(r3))))))))))',
  '(G((q3) -> ((F(bool_exp3)) -> ((!(bool_exp3)) U ((r3) | ((!(bool_exp3)) & (bool_exp4) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp5) & (X((!(bool_exp3)) U ((!(bool_exp3)) & (bool_exp6)))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | ((!(r3)) & (X((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))))))) U ((r3) | (G((bool_exp3) -> (((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | ((!(r3)) & (X((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4) & (X((TRUE) U ((!(r3)) & (bool_exp5) & (X((TRUE) U (bool_exp6))))))))) U ((r3) | (G((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4) & (X((TRUE) U ((!(r3)) & (bool_exp5) & (X((TRUE) U (bool_exp6))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))))))))) | ((!(bool_cnt_exp3)) & (!(r3)) & (X((!(bool_cnt_exp3)) & (!(r3)))) & (X(X((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X(((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))) | ((!(bool_cnt_exp4)) & (X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6))))))))) | ((!(bool_cnt_exp4)) & (X(!(bool_cnt_exp4))) & (X(X((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((bool_exp6) | ((!(bool_cnt_exp5)) & (X(bool_exp6))) | ((!(bool_cnt_exp5)) & (X(!(bool_cnt_exp5))) & (X(X(bool_exp6)))))))))))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | ((!(r3)) & (X((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))))))) U ((r3) | (G((bool_exp3) -> (((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))) | ((!(r3)) & (X((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))))))))) | ((!(r3)) & (X(!(r3))) & (X(X((!(r3)) & (bool_exp4) & (X(((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))) | (X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6))))))) | (X(X((!(r3)) & (bool_exp5) & (X((bool_exp6) | (X(bool_exp6)) | (X(X(bool_exp6)))))))))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))) U ((r3) | (G((bool_exp3) -> (((!(bool_cnt_exp3)) & (!(r3))) U ((!(bool_cnt_exp4)) & (!(r3)) & (bool_exp4) & (X((!(bool_cnt_exp4)) U ((!(bool_cnt_exp5)) & (!(r3)) & (bool_exp5) & (X((!(bool_cnt_exp5)) U (bool_exp6))))))))))))))',
  '(G((q3) -> (((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4) & (X((TRUE) U ((!(r3)) & (bool_exp5) & (X((TRUE) U (bool_exp6))))))))) U ((r3) | (G((bool_exp3) -> ((!(r3)) U ((!(r3)) & (bool_exp4) & (X((TRUE) U ((!(r3)) & (bool_exp5) & (X((TRUE) U (bool_exp6))))))))))))))']]


universal_var_names = ['r1', 'q1', 'q2', 'r2', 'q3', 'r3']

# run proxy generation

In [ ]:
%%time

all_f_list, all_shift_step_list, all_sign_list, proxy_list, proxy_start_step_list, all_constraint_aut = proxy_gen.top_proxy_search(all_f_list,universal_var_names)
print("proxies generated!")

In [ ]:
%%time
assert proxy_gen.verify_proxies(all_f_list,all_shift_step_list,all_sign_list, proxy_list, proxy_start_step_list, all_constraint_aut)
print("proxies verified!")